# LAB 07 — SELF-ATTENTION

## 17. Self-Attention (Cơ chế tự chú ý)

Self-Attention là cơ chế cho phép mỗi token xác định mức độ liên quan với các token khác trong cùng một chuỗi.

Mục đích:
- Xác định mối quan hệ giữa các token.
- Giúp mỗi token thu thập thông tin từ những token khác.
- Tạo ra biểu diễn mới chứa thông tin ngữ cảnh.

Ví dụ:

Câu: "The animal didn't cross the street because it was tired."

Từ "it" cần xác định mối liên hệ với những từ khác để hiểu ngữ cảnh của câu.

### Prediction

1. Một token có thể chú ý đến chính nó và các token khác trong chuỗi không?
2. Nếu hai token có mức độ liên quan cao, trọng số chú ý giữa chúng được dự đoán sẽ lớn hay nhỏ?
3. Sau Self-Attention, biểu diễn của một token có còn phụ thuộc hoàn toàn vào embedding ban đầu không?

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print("PyTorch:", torch.__version__)

PyTorch: 2.14.1+cpu


In [2]:
x = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0]
])

print("Input:")
print(x)

print("Input shape:", x.shape)

Input:
tensor([[1., 0.],
        [0., 1.],
        [1., 1.]])
Input shape: torch.Size([3, 2])


In [3]:
scores = torch.matmul(x, x.T)

print("Attention scores:")
print(scores)
print("Scores shape:", scores.shape)

Attention scores:
tensor([[1., 0., 1.],
        [0., 1., 1.],
        [1., 1., 2.]])
Scores shape: torch.Size([3, 3])


### Nhận xét kết quả (Observation)

- Kích thước ma trận Attention Score: (3, 3).
- Mỗi hàng biểu diễn điểm số của một token khi so sánh với tất cả token trong chuỗi.
- Các phần tử trên đường chéo chính thể hiện điểm số khi token so sánh với chính nó.
- Token 1 và Token 2 có điểm số bằng 0 vì hai vector vuông góc.

Kết quả cho thấy phép nhân ma trận tạo ra điểm số tương đồng giữa các token. Tuy nhiên, các giá trị này chưa phải xác suất chú ý mà cần được chuẩn hóa bằng hàm Softmax.

In [4]:
attention_weights = F.softmax(scores, dim=-1)

print("Attention weights:")
print(attention_weights)

print("Row sums:")
print(attention_weights.sum(dim=-1))

Attention weights:
tensor([[0.4223, 0.1554, 0.4223],
        [0.1554, 0.4223, 0.4223],
        [0.2119, 0.2119, 0.5761]])
Row sums:
tensor([1., 1., 1.])


### Nhận xét kết quả (Observation)

- Các trọng số chú ý đều nằm trong khoảng từ 0 đến 1.
- Tổng trọng số trên mỗi hàng đều bằng 1.
- Token 3 có trọng số chú ý lớn nhất đối với chính nó, đạt 0.5761.

Hàm Softmax chuyển ma trận Attention Score thành ma trận Attention Weights. Các giá trị sau chuẩn hóa thể hiện mức độ chú ý tương đối giữa các token trong chuỗi.

Mỗi hàng có tổng bằng 1, cho phép xem các trọng số như một phân phối xác suất.

In [5]:
values = x

attention_output = torch.matmul(attention_weights, values)

print("Attention output:")
print(attention_output)

print("Output shape:", attention_output.shape)

Attention output:
tensor([[0.8446, 0.5777],
        [0.5777, 0.8446],
        [0.7881, 0.7881]])
Output shape: torch.Size([3, 2])


### Nhận xét kết quả (Observation)

- Kích thước đầu vào: (3, 2)
- Kích thước đầu ra: (3, 2)
- Đầu ra của mỗi token là sự kết hợp có trọng số của các vector Value.

Kết quả thực nghiệm cho thấy Self-Attention tạo ra biểu diễn mới cho mỗi token bằng cách tổng hợp thông tin từ các token trong cùng chuỗi.

Kích thước vector được giữ nguyên, nhưng nội dung biểu diễn đã thay đổi và chứa thêm thông tin ngữ cảnh.

## Query, Key, Value (Q, K, V)

Trong cơ chế Self-Attention, mỗi token được biến đổi thành ba vector:

- Query (Q): Đại diện cho thông tin mà token đang tìm kiếm.
- Key (K): Đại diện cho thông tin mà token cung cấp để được so sánh.
- Value (V): Chứa thông tin được sử dụng để tạo biểu diễn đầu ra.

Ba vector được tạo ra thông qua các phép biến đổi tuyến tính có tham số được học trong quá trình huấn luyện.

Cách hoạt động:

1. Query được so sánh với các Key để tính điểm chú ý.
2. Softmax chuẩn hóa điểm chú ý thành trọng số.
3. Các trọng số được sử dụng để kết hợp những vector Value.

Mục đích là giúp mỗi token xác định thông tin cần thu thập từ những token khác.

In [6]:
d_model = 4
d_k = 4

x = torch.randn(3, d_model)

W_q = nn.Linear(d_model, d_k, bias=False)
W_k = nn.Linear(d_model, d_k, bias=False)
W_v = nn.Linear(d_model, d_k, bias=False)

Q = W_q(x)
K = W_k(x)
V = W_v(x)

print("Input shape:", x.shape)
print("Query shape:", Q.shape)
print("Key shape:", K.shape)
print("Value shape:", V.shape)

Input shape: torch.Size([3, 4])
Query shape: torch.Size([3, 4])
Key shape: torch.Size([3, 4])
Value shape: torch.Size([3, 4])


### Nhận xét kết quả (Observation)

- Kích thước đầu vào: (3, 4).
- Kích thước Query: (3, 4).
- Kích thước Key: (3, 4).
- Kích thước Value: (3, 4).

Kết quả thực nghiệm khớp với dự đoán.

Ba vector Query, Key và Value được tạo từ cùng dữ liệu đầu vào thông qua ba phép biến đổi tuyến tính độc lập. Mặc dù có cùng kích thước, giá trị của chúng có thể khác nhau do sử dụng các ma trận trọng số riêng biệt.

In [7]:
print("Query:")
print(Q)

print("Key:")
print(K)

print("Value:")
print(V)

print("Q equals K:", torch.allclose(Q, K))
print("Q equals V:", torch.allclose(Q, V))
print("K equals V:", torch.allclose(K, V))

Query:
tensor([[-0.0856, -0.4070, -0.0362,  0.5559],
        [ 0.4288,  0.5883,  0.0901, -0.4989],
        [ 0.2817,  0.4332, -0.0076, -0.1014]], grad_fn=<MmBackward0>)
Key:
tensor([[ 0.2796,  0.0950,  0.0302, -0.0582],
        [-0.4896,  0.0813, -0.1470, -0.4243],
        [-0.4158, -0.0625,  0.1122, -0.3503]], grad_fn=<MmBackward0>)
Value:
tensor([[-0.1034,  0.2565, -0.2230, -0.3108],
        [-0.1587, -0.1432,  0.5118,  0.7807],
        [-0.0794,  0.3414,  0.5124,  0.6686]], grad_fn=<MmBackward0>)
Q equals K: False
Q equals V: False
K equals V: False


### Nhận xét kết quả (Observation)

- Q equals K: False
- Q equals V: False
- K equals V: False

Kết quả thực nghiệm khớp với dự đoán.

Query, Key và Value có giá trị khác nhau vì được tạo thông qua ba phép biến đổi tuyến tính độc lập với các ma trận trọng số riêng.

Mỗi vector đảm nhiệm một vai trò khác nhau trong cơ chế Self-Attention: Query dùng để tìm kiếm thông tin, Key dùng để tính mức độ liên quan và Value chứa thông tin được tổng hợp vào đầu ra.

## Scaled Dot-Product Attention

Scaled Dot-Product Attention là cơ chế tính toán mức độ chú ý giữa các token dựa trên ba vector Query, Key và Value.

Quy trình:

1. Nhân Query với Key chuyển vị để tính điểm chú ý.
2. Chia điểm chú ý cho căn bậc hai của số chiều Key.
3. Áp dụng Softmax để chuẩn hóa điểm chú ý.
4. Nhân trọng số chú ý với Value để tạo đầu ra.

Công thức:

Attention(Q, K, V) = Softmax(QK^T / sqrt(d_k)) V

Trong đó:
- Q: Query.
- K: Key.
- V: Value.
- d_k: Số chiều của Key.

### Prediction

1. Nếu d_k tăng, việc chia cho căn bậc hai của d_k có tác dụng gì đối với điểm chú ý?
2. Sau Softmax, tổng trọng số chú ý trên mỗi hàng có bằng 1 không?
3. Kích thước đầu ra Attention có phụ thuộc vào kích thước của Value không?|

In [9]:
scores = torch.matmul(Q, K.transpose(-2, -1))

print("Raw attention scores:")
print(scores)
print("Scores shape:", scores.shape)

Raw attention scores:
tensor([[-0.0960, -0.2218, -0.1378],
        [ 0.2075,  0.0363, -0.0302],
        [ 0.1256, -0.0585, -0.1095]], grad_fn=<MmBackward0>)
Scores shape: torch.Size([3, 3])


In [10]:
scaled_scores = scores / (d_k ** 0.5)

print("Scaled attention scores:")
print(scaled_scores)

Scaled attention scores:
tensor([[-0.0480, -0.1109, -0.0689],
        [ 0.1038,  0.0182, -0.0151],
        [ 0.0628, -0.0293, -0.0548]], grad_fn=<DivBackward0>)


### Nhận xét kết quả (Observation)

- Raw attention scores có kích thước (3, 3).
- Scaled attention scores có kích thước (3, 3).
- Với d_k = 4, các điểm chú ý được chia cho 2.

Phép chia cho căn bậc hai của d_k giúp kiểm soát độ lớn của điểm chú ý trước khi áp dụng Softmax. Điều này hạn chế việc Softmax tạo ra phân phối quá tập trung khi các điểm số có giá trị lớn.

In [11]:
attention_weights = F.softmax(scaled_scores, dim=-1)

attention_output = torch.matmul(attention_weights, V)

print("Attention weights:")
print(attention_weights)

print("Row sums:")
print(attention_weights.sum(dim=-1))

print("Attention output:")
print(attention_output)

print("Output shape:", attention_output.shape)

Attention weights:
tensor([[0.3427, 0.3218, 0.3356],
        [0.3564, 0.3272, 0.3164],
        [0.3570, 0.3256, 0.3174]], grad_fn=<SoftmaxBackward0>)
Row sums:
tensor([1., 1., 1.], grad_fn=<SumBackward1>)
Attention output:
tensor([[-0.1131,  0.1564,  0.2602,  0.3691],
        [-0.1139,  0.1526,  0.2501,  0.3562],
        [-0.1138,  0.1533,  0.2497,  0.3554]], grad_fn=<MmBackward0>)
Output shape: torch.Size([3, 4])


### Nhận xét kết quả (Observation)

- Attention weights có kích thước (3, 3).
- Tất cả trọng số đều nằm trong khoảng [0, 1].
- Tổng trọng số trên mỗi hàng bằng 1.
- Attention output có kích thước (3, 4).

Softmax chuẩn hóa các điểm chú ý thành một phân phối trọng số. Mỗi token sử dụng phân phối này để tổng hợp thông tin từ các vector Value.

Kích thước đầu ra bằng kích thước của V, cho thấy Attention giữ nguyên số lượng token và số chiều biểu diễn của Value.

### Implementation

Hàm Scaled Dot-Product Attention thực hiện đầy đủ 4 bước:

1. Tính QK^T để xác định điểm chú ý.
2. Chia điểm chú ý cho căn bậc hai của d_k.
3. Áp dụng Softmax để tạo Attention Weights.
4. Nhân Attention Weights với V để tạo đầu ra.

Hàm trả về cả Attention Output và Attention Weights.

In [12]:
def scaled_dot_product_attention(Q, K, V):
    d_k = K.size(-1)

    scores = torch.matmul(Q, K.transpose(-2, -1))
    scaled_scores = scores / (d_k ** 0.5)

    attention_weights = F.softmax(scaled_scores, dim=-1)
    output = torch.matmul(attention_weights, V)

    return output, attention_weights

In [13]:
output, weights = scaled_dot_product_attention(Q, K, V)

print("Output shape:", output.shape)
print("Weights shape:", weights.shape)

print("Output:")
print(output)

print("Weights:")
print(weights)

print("Row sums:", weights.sum(dim=-1))

Output shape: torch.Size([3, 4])
Weights shape: torch.Size([3, 3])
Output:
tensor([[-0.1131,  0.1564,  0.2602,  0.3691],
        [-0.1139,  0.1526,  0.2501,  0.3562],
        [-0.1138,  0.1533,  0.2497,  0.3554]], grad_fn=<MmBackward0>)
Weights:
tensor([[0.3427, 0.3218, 0.3356],
        [0.3564, 0.3272, 0.3164],
        [0.3570, 0.3256, 0.3174]], grad_fn=<SoftmaxBackward0>)
Row sums: tensor([1., 1., 1.], grad_fn=<SumBackward1>)


### Nhận xét kết quả (Observation)

- Output shape: (3, 4).
- Weights shape: (3, 3).
- Tổng trọng số trên mỗi hàng: 1.

Kết quả từ hàm Scaled Dot-Product Attention trùng khớp với kết quả tính toán từng bước trước đó.

Điều này xác nhận hàm đã thực hiện đúng quy trình: tính điểm chú ý, Scaling, Softmax và tổng hợp thông tin từ Value.

Hàm có thể tái sử dụng với các đầu vào Q, K, V có kích thước phù hợp.

### Prediction

**Prediction 1:** Token ở vị trí 1 không thể chú ý đến token ở vị trí 3.

**Prediction 2:** Những vị trí bị Mask sẽ có trọng số Attention bằng 0 sau Softmax.

**Prediction 3:** Ma trận Mask có dạng tam giác dưới, vì mỗi token chỉ được phép chú ý đến chính nó và các token đứng trước nó.

### Reason

Causal Masking được thiết kế để ngăn mô hình sử dụng thông tin từ tương lai. Điều này giúp bảo đảm quá trình dự đoán token tiếp theo chỉ dựa trên những thông tin đã xuất hiện

In [14]:
scores = torch.tensor([
    [2.0, 1.0, 3.0, 4.0],
    [1.0, 3.0, 2.0, 4.0],
    [2.0, 1.0, 4.0, 3.0],
    [3.0, 2.0, 1.0, 4.0]
])

print("Attention scores:")
print(scores)

Attention scores:
tensor([[2., 1., 3., 4.],
        [1., 3., 2., 4.],
        [2., 1., 4., 3.],
        [3., 2., 1., 4.]])


In [15]:
sequence_length = scores.size(0)

mask = torch.tril(torch.ones(sequence_length, sequence_length))

print("Causal mask:")
print(mask)

Causal mask:
tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])


In [16]:
masked_scores = scores.masked_fill(mask == 0, float('-inf'))

print("Masked attention scores:")
print(masked_scores)

Masked attention scores:
tensor([[2., -inf, -inf, -inf],
        [1., 3., -inf, -inf],
        [2., 1., 4., -inf],
        [3., 2., 1., 4.]])


In [17]:
masked_weights = F.softmax(masked_scores, dim=-1)

print("Masked attention weights:")
print(masked_weights)

print("Row sums:")
print(masked_weights.sum(dim=-1))

Masked attention weights:
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.1192, 0.8808, 0.0000, 0.0000],
        [0.1142, 0.0420, 0.8438, 0.0000],
        [0.2369, 0.0871, 0.0321, 0.6439]])
Row sums:
tensor([1.0000, 1.0000, 1.0000, 1.0000])


### Nhận xét kết quả (Observation)

- Ma trận Causal Mask có dạng tam giác dưới.
- Các vị trí bị Mask có Attention Weight bằng 0.
- Tổng trọng số trên mỗi hàng vẫn bằng 1.

Causal Masking ngăn mỗi token chú ý đến những token xuất hiện phía sau nó trong sequence.

Sau khi áp dụng Mask, Softmax chỉ phân phối trọng số trên các vị trí được phép. Điều này bảo đảm mô hình không sử dụng thông tin từ tương lai trong quá trình sinh văn bản.

### Prediction

**Prediction 1:** Nếu sử dụng 4 Attention Head, mô hình sẽ thực hiện 4 phép tính Attention độc lập.

**Prediction 2:** Nếu mỗi Head tạo ra vector 4 chiều, việc Concatenate 4 Head sẽ tạo ra vector 16 chiều.

**Prediction 3:** Sau Linear Projection, kích thước đầu ra có thể trở về d_model = 16.

### Reason

Mỗi Attention Head có các ma trận trọng số Q, K, V riêng. Các Head xử lý thông tin song song, sau đó kết quả được ghép lại và biến đổi tuyến tính.

In [18]:
batch_size = 2
sequence_length = 5
d_model = 16
num_heads = 4
d_k = d_model // num_heads

x = torch.randn(batch_size, sequence_length, d_model)

print("Input shape:", x.shape)
print("Number of heads:", num_heads)
print("Dimension per head:", d_k)

Input shape: torch.Size([2, 5, 16])
Number of heads: 4
Dimension per head: 4


In [19]:
W_q = nn.Linear(d_model, d_model)
W_k = nn.Linear(d_model, d_model)
W_v = nn.Linear(d_model, d_model)

Q = W_q(x)
K = W_k(x)
V = W_v(x)

print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)

Q shape: torch.Size([2, 5, 16])
K shape: torch.Size([2, 5, 16])
V shape: torch.Size([2, 5, 16])


In [20]:
def split_heads(tensor, num_heads):
    batch_size, sequence_length, d_model = tensor.shape

    tensor = tensor.reshape(
        batch_size,
        sequence_length,
        num_heads,
        d_model // num_heads
    )

    return tensor.transpose(1, 2)

Q_heads = split_heads(Q, num_heads)
K_heads = split_heads(K, num_heads)
V_heads = split_heads(V, num_heads)

print("Q heads shape:", Q_heads.shape)
print("K heads shape:", K_heads.shape)
print("V heads shape:", V_heads.shape)

Q heads shape: torch.Size([2, 4, 5, 4])
K heads shape: torch.Size([2, 4, 5, 4])
V heads shape: torch.Size([2, 4, 5, 4])


### Nhận xét kết quả (Observation)

- Q, K, V ban đầu có kích thước (2, 5, 16).
- Sau khi tách Head, cả ba tensor có kích thước (2, 4, 5, 4).

Phép reshape và transpose đã chuyển chiều biểu diễn 16 thành 4 nhóm, mỗi nhóm có 4 chiều.

Nhờ đó, 4 Attention Head có thể thực hiện phép tính Attention độc lập trên cùng một sequence.

In [21]:
scores = torch.matmul(Q_heads, K_heads.transpose(-2, -1))
scaled_scores = scores / (d_k ** 0.5)

attention_weights = F.softmax(scaled_scores, dim=-1)

head_outputs = torch.matmul(attention_weights, V_heads)

print("Attention weights shape:", attention_weights.shape)
print("Head outputs shape:", head_outputs.shape)

Attention weights shape: torch.Size([2, 4, 5, 5])
Head outputs shape: torch.Size([2, 4, 5, 4])


In [22]:
head_outputs = head_outputs.transpose(1, 2)

concat_output = head_outputs.reshape(
    batch_size,
    sequence_length,
    d_model
)

print("Concatenated output shape:", concat_output.shape)

Concatenated output shape: torch.Size([2, 5, 16])


In [23]:
W_o = nn.Linear(d_model, d_model)

mha_output = W_o(concat_output)

print("Final MHA output shape:", mha_output.shape)
print("Input shape:", x.shape)

Final MHA output shape: torch.Size([2, 5, 16])
Input shape: torch.Size([2, 5, 16])


In [24]:
print("Input shape:", x.shape)
print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)
print("Attention weights shape:", attention_weights.shape)
print("Concatenated output shape:", concat_output.shape)
print("Final MHA output shape:", mha_output.shape)

Input shape: torch.Size([2, 5, 16])
Q shape: torch.Size([2, 5, 16])
K shape: torch.Size([2, 5, 16])
V shape: torch.Size([2, 5, 16])
Attention weights shape: torch.Size([2, 4, 5, 5])
Concatenated output shape: torch.Size([2, 5, 16])
Final MHA output shape: torch.Size([2, 5, 16])


### Nhận xét kết quả (Observation)

- Input shape: (2, 5, 16).
- Number of heads: 4.
- Dimension per head: 4.
- Concatenated output shape: (2, 5, 16).
- Final MHA output shape: (2, 5, 16).

Bốn Attention Head thực hiện tính toán độc lập và song song. Kết quả được Concatenate, sau đó đưa qua Linear Projection để kết hợp thông tin giữa các Head.

Kích thước đầu ra cuối cùng được giữ nguyên so với đầu vào, mặc dù thông tin đã được xử lý qua nhiều Attention Head.

### Nhận xét kết quả (Observation)

- Input shape: (2, 5, 16).
- Number of heads: 4.
- Dimension per head: 4.
- Attention weights shape: (2, 4, 5, 5).
- Concatenated output shape: (2, 5, 16).
- Final MHA output shape: (2, 5, 16).

Bốn Attention Head thực hiện tính toán độc lập và song song trên cùng một sequence. Kết quả được Concatenate, sau đó đưa qua Linear Projection để kết hợp thông tin giữa các Head.

Kích thước đầu ra cuối cùng được giữ nguyên so với đầu vào. Multi-Head Attention cho phép mô hình khai thác nhiều mối quan hệ khác nhau giữa các token thông qua nhiều không gian biểu diễn.

In [25]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        batch_size, sequence_length, _ = x.shape

        x = x.reshape(
            batch_size,
            sequence_length,
            self.num_heads,
            self.d_k
        )

        return x.transpose(1, 2)

    def forward(self, x):
        Q = self.split_heads(self.W_q(x))
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / (self.d_k ** 0.5)

        weights = F.softmax(scores, dim=-1)

        output = torch.matmul(weights, V)
        output = output.transpose(1, 2)

        batch_size, sequence_length, _, _ = output.shape
        output = output.reshape(batch_size, sequence_length, self.d_model)

        output = self.W_o(output)

        return output, weights

In [26]:
mha = MultiHeadAttention(d_model=16, num_heads=4)

output, weights = mha(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Attention weights shape:", weights.shape)

Input shape: torch.Size([2, 5, 16])
Output shape: torch.Size([2, 5, 16])
Attention weights shape: torch.Size([2, 4, 5, 5])


In [27]:
total_params = sum(p.numel() for p in mha.parameters())

print("Total parameters:", total_params)

Total parameters: 1088


In [28]:
mha.zero_grad()

loss = output.sum()
loss.backward()

print("Q weight gradient exists:", mha.W_q.weight.grad is not None)
print("K weight gradient exists:", mha.W_k.weight.grad is not None)
print("V weight gradient exists:", mha.W_v.weight.grad is not None)
print("Output weight gradient exists:", mha.W_o.weight.grad is not None)

Q weight gradient exists: True
K weight gradient exists: True
V weight gradient exists: True
Output weight gradient exists: True


### Nhận xét kết quả (Observation)

- Input shape: (2, 5, 16).
- Output shape: (2, 5, 16).
- Attention weights shape: (2, 4, 5, 5).
- Total parameters: 1.088.
- Gradient của cả 4 Linear Layer đều được tính toán thành công.

Class MultiHeadAttention đã triển khai đầy đủ quy trình Multi-Head Attention, bao gồm tạo Q, K, V, tách Head, tính Scaled Dot-Product Attention, Concatenate và Linear Projection.

Kết quả Backpropagation xác nhận các tham số có thể nhận Gradient, cho phép mô hình được tối ưu hóa trong quá trình huấn luyện.

### Prediction

**Prediction 1:** Transformer Encoder Block sẽ giữ nguyên kích thước đầu vào và đầu ra.

**Prediction 2:** Residual Connection yêu cầu hai tensor được cộng phải có cùng kích thước.

**Prediction 3:** FFN xử lý từng token độc lập, không làm thay đổi số lượng token trong sequence.

### Reason

Multi-Head Attention và FFN đều được thiết kế để giữ nguyên d_model ở đầu ra. Residual Connection thực hiện phép cộng giữa đầu vào và kết quả xử lý nên hai tensor phải có cùng kích thước.

In [29]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)

        return x

In [30]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)

        self.ffn = FeedForwardNetwork(d_model, d_ff)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attention_output, weights = self.attention(x)
        x = self.norm1(x + attention_output)

        ffn_output = self.ffn(x)
        output = self.norm2(x + ffn_output)

        return output, weights

In [31]:
encoder = TransformerEncoderBlock(
    d_model=16,
    num_heads=4,
    d_ff=64
)

output, weights = encoder(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Attention weights shape:", weights.shape)

Input shape: torch.Size([2, 5, 16])
Output shape: torch.Size([2, 5, 16])
Attention weights shape: torch.Size([2, 4, 5, 5])


In [32]:
print("Output mean:")
print(output.mean(dim=-1))

print("Output std:")
print(output.std(dim=-1, unbiased=False))

Output mean:
tensor([[-2.9802e-08,  1.4901e-08,  7.4506e-09,  8.3819e-09,  0.0000e+00],
        [-7.4506e-09, -2.2352e-08,  3.7253e-09, -2.2352e-08, -1.1176e-08]],
       grad_fn=<MeanBackward1>)
Output std:
tensor([[1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000]], grad_fn=<StdBackward0>)


In [33]:
encoder.zero_grad()

loss = output.sum()
loss.backward()

print("Attention gradient exists:",
      encoder.attention.W_q.weight.grad is not None)

print("FFN gradient exists:",
      encoder.ffn.linear1.weight.grad is not None)

print("LayerNorm 1 gradient exists:",
      encoder.norm1.weight.grad is not None)

print("LayerNorm 2 gradient exists:",
      encoder.norm2.weight.grad is not None)

Attention gradient exists: True
FFN gradient exists: True
LayerNorm 1 gradient exists: True
LayerNorm 2 gradient exists: True


### Nhận xét kết quả (Observation)

- Input shape: (2, 5, 16).
- Output shape: (2, 5, 16).
- Attention weights shape: (2, 4, 5, 5).

Transformer Encoder Block giữ nguyên kích thước đầu vào và đầu ra.

Multi-Head Attention giúp các token trao đổi thông tin. Residual Connection kết hợp thông tin ban đầu với kết quả xử lý. Layer Normalization chuẩn hóa biểu diễn, trong khi FFN tiếp tục xử lý đặc trưng của từng token.

Kết quả cho thấy các thành phần có thể kết hợp thành một Encoder Block hoàn chỉnh.

In [35]:
encoder.zero_grad()

output, weights = encoder(x)

weights_vector = torch.arange(1, 17, dtype=output.dtype).view(1, 1, 16)
loss = (output * weights_vector).sum()

loss.backward()

print("Attention gradient norm:", encoder.attention.W_q.weight.grad.norm().item())
print("FFN gradient norm:", encoder.ffn.linear1.weight.grad.norm().item())
print("LayerNorm 1 gradient norm:", encoder.norm1.weight.grad.norm().item())
print("LayerNorm 2 gradient norm:", encoder.norm2.weight.grad.norm().item())

Attention gradient norm: 9.910652160644531
FFN gradient norm: 96.27500915527344
LayerNorm 1 gradient norm: 43.303409576416016
LayerNorm 2 gradient norm: 153.0465545654297


### Nhận xét kết quả (Observation)

- Input shape: (2, 5, 16).
- Output shape: (2, 5, 16).
- Attention weights shape: (2, 4, 5, 5).
- Gradient của Attention, FFN và hai LayerNorm đều khác 0.

Transformer Encoder Block giữ nguyên kích thước đầu vào và đầu ra.

Multi-Head Attention giúp các token trao đổi thông tin. Residual Connection kết hợp thông tin ban đầu với kết quả xử lý. Layer Normalization chuẩn hóa biểu diễn, trong khi FFN tiếp tục xử lý đặc trưng của từng token.

Kết quả backpropagation xác nhận gradient có thể lan truyền qua toàn bộ Encoder Block. Như vậy, các thành phần đã được kết hợp thành một khối Transformer Encoder hoàn chỉnh.

## Stacking Transformer Encoder Blocks

Một Transformer Encoder hoàn chỉnh có thể được xây dựng bằng cách xếp chồng nhiều Transformer Encoder Block.

Mỗi Block tiếp tục xử lý biểu diễn của các token từ Block trước đó.

Các Block có cùng kích thước đầu vào và đầu ra, vì vậy có thể kết nối liên tiếp với nhau.

Mục tiêu thí nghiệm:
- Xây dựng Encoder gồm nhiều Block.
- Kiểm tra kích thước đầu vào và đầu ra.
- Kiểm tra số lượng tham số khi tăng số Block.

In [36]:
class TransformerEncoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])

    def forward(self, x):
        attention_weights = []

        for layer in self.layers:
            x, weights = layer(x)
            attention_weights.append(weights)

        return x, attention_weights

In [37]:
encoder = TransformerEncoder(
    d_model=16,
    num_heads=4,
    d_ff=64,
    num_layers=3
)

x = torch.randn(2, 5, 16)

output, attention_weights = encoder(x)

total_params = sum(p.numel() for p in encoder.parameters())

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Number of encoder blocks:", len(encoder.layers))
print("Number of attention weight tensors:", len(attention_weights))
print("Total parameters:", total_params)

Input shape: torch.Size([2, 5, 16])
Output shape: torch.Size([2, 5, 16])
Number of encoder blocks: 3
Number of attention weight tensors: 3
Total parameters: 9840


### Nhận xét kết quả (Observation)

- Kích thước đầu vào: (2, 5, 16).
- Kích thước đầu ra: (2, 5, 16).
- Số Encoder Blocks: 3.
- Số tensor Attention Weights: 3.
- Tổng số tham số: 9.840.

Ba Transformer Encoder Blocks được xếp chồng liên tiếp. Mỗi block tiếp nhận đầu ra của block trước và tiếp tục xử lý biểu diễn của các token.

Kích thước đầu vào và đầu ra không thay đổi, trong khi thông tin biểu diễn được cập nhật qua từng block. Mỗi block có một cơ chế Multi-Head Attention riêng, vì vậy tạo ra một tensor Attention Weights riêng.

Kết quả xác nhận có thể xây dựng Transformer Encoder gồm nhiều block mà vẫn giữ nguyên cấu trúc tensor đầu vào.

## 24. Gradient Flow Through Multiple Encoder Blocks

### Mục tiêu

Kiểm tra khả năng lan truyền gradient qua nhiều Transformer Encoder Blocks.

### Prediction

- Gradient có thể truyền ngược từ Encoder Block cuối về Encoder Block đầu.
- Các tham số Attention và FFN trong mỗi block đều có thể nhận gradient.
- Khi tăng số lượng Encoder Blocks, mô hình có thêm nhiều tầng tham số cần được tối ưu.

In [39]:
stacked_encoder = nn.ModuleList([
    TransformerEncoderBlock(d_model=16, num_heads=4, d_ff=64)
    for _ in range(3)
])

stacked_encoder.zero_grad()

output = x

for encoder in stacked_encoder:
    output, _ = encoder(output)

weights = torch.arange(1, 17, dtype=output.dtype).view(1, 1, 16)
loss = (output * weights).sum()
loss.backward()

for i, encoder in enumerate(stacked_encoder):
    print(f"Encoder Block {i + 1}:")
    print("Attention gradient:",
          encoder.attention.W_q.weight.grad.norm().item())
    print("FFN gradient:",
          encoder.ffn.linear1.weight.grad.norm().item())
    print()

Encoder Block 1:
Attention gradient: 16.876985549926758
FFN gradient: 111.7951889038086

Encoder Block 2:
Attention gradient: 18.245573043823242
FFN gradient: 107.31675720214844

Encoder Block 3:
Attention gradient: 19.11680030822754
FFN gradient: 88.94190216064453



### Nhận xét kết quả (Observation)

Cả 3 Encoder Blocks đều nhận được gradient thông qua quá trình Backpropagation.

- Block 1: Attention = 16.8770, FFN = 111.7952.
- Block 2: Attention = 18.2456, FFN = 107.3168.
- Block 3: Attention = 19.1168, FFN = 88.9419.

Các gradient đều khác 0, cho thấy tín hiệu học có thể truyền từ đầu ra về các tầng trước đó.

Kết quả xác nhận các tham số Attention và FFN trong cả 3 Encoder Blocks đều tham gia vào quá trình tối ưu mô hình.